# EDA: `Measurements.csv`

Exploratory data analysis of the wellness screening measurements dataset.

**Focus (per Pulse80 data engineering role):** grain integrity, missing-data patterns, validation/contract
coverage, unit consistency, and outlier/distribution diagnostics — the checks that determine whether this
table is safe to key into downstream analytical models, rather than presentation-ready charts.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

df = pd.read_csv('Measurements.csv')
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(10)

## 1. Structure & data types

The table is in **long/EAV format**: one row per `(screening_id, metric_code)` observation, rather than one
row per screening. This is a common source-system pattern — worth confirming before assuming any grain.

In [ ]:
df.info()
print()
print("Dtypes:")
print(df.dtypes)

## 2. Missing data patterns

Diagnostic check before anything else: which columns are populated, and does `text_value` ever carry data
(distinguishing numeric vs. categorical/text metrics)?

In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
})
missing_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
msno.bar(df, ax=ax, color='#4C72B0')
plt.title('Column completeness')
plt.tight_layout()
plt.show()

**Finding:** `text_value` is 100% null across all 264 rows. Every metric in this extract is numeric —
so either (a) this source never emits categorical/free-text measurements, or (b) text-valued metrics are
routed elsewhere upstream. Worth confirming against the data contract rather than assuming the column is
dead weight.

## 3. Grain integrity

Checking the primary key and the natural composite key before trusting any join or aggregation downstream.

In [ ]:
# Primary key: measurement_id should be unique
dupe_ids = df['measurement_id'].duplicated().sum()
print(f"Duplicate measurement_id values: {dupe_ids}")

# Natural key: one (screening_id, metric_code) pair should appear once per screening event
dupe_natural_key = df.duplicated(subset=['screening_id', 'metric_code']).sum()
print(f"Duplicate (screening_id, metric_code) pairs: {dupe_natural_key}")

print(f"\nUnique screening_id count: {df['screening_id'].nunique()}")
print(f"Unique metric_code count: {df['metric_code'].nunique()}")
print(f"Rows per screening_id (should be consistent if every screening captures the full panel):")
print(df.groupby('screening_id').size().value_counts())

## 4. Metric coverage & validation status

Which metrics are present, how often, and does `validation_status` show any failed/pending records that
need to be excluded before aggregation?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metric_counts = df['metric_code'].value_counts()
sns.barplot(x=metric_counts.values, y=metric_counts.index, ax=axes[0], color='#4C72B0')
axes[0].set_title('Row count by metric_code')
axes[0].set_xlabel('count')

status_counts = df['validation_status'].value_counts()
sns.barplot(x=status_counts.values, y=status_counts.index, ax=axes[1], color='#DD8452')
axes[1].set_title('validation_status distribution')
axes[1].set_xlabel('count')

plt.tight_layout()
plt.show()
print(status_counts)

## 5. Unit consistency per metric

A single metric should map to exactly one unit. Any metric with >1 distinct unit is a red flag for
inconsistent source capture (e.g. imperial vs. metric height/weight) that would silently corrupt an
aggregate if unresolved.

In [ ]:
unit_check = df.groupby('metric_code')['unit'].agg(['nunique', lambda x: sorted(x.unique())])
unit_check.columns = ['n_distinct_units', 'units']
unit_check

## 6. Timestamp discipline

Confirming `captured_at` parses cleanly, checking timezone representation (UTC discipline matters for any
downstream `rolling()`/`shift()` window logic), and looking at the capture time range.

In [ ]:
df['captured_at_parsed'] = pd.to_datetime(df['captured_at'], utc=True, errors='coerce')
n_unparseable = df['captured_at_parsed'].isna().sum()
print(f"Unparseable timestamps: {n_unparseable}")
print(f"Range: {df['captured_at_parsed'].min()}  ->  {df['captured_at_parsed'].max()}")

# Confirm every raw string is UTC ('Z' suffix) rather than mixed offsets
non_utc_suffix = (~df['captured_at'].str.endswith('Z')).sum()
print(f"Timestamps not using explicit 'Z' (UTC) suffix: {non_utc_suffix}")

## 7. Reshape to wide (screening-level) grain

For distribution/outlier/correlation analysis it's more useful to pivot to one row per `screening_id`,
mirroring how this would land as a fact/dimension table downstream.

In [ ]:
wide = df.pivot_table(index='screening_id', columns='metric_code', values='numeric_value', aggfunc='first')
print(f"Wide shape: {wide.shape}")
wide.head(10)

In [ ]:
wide.describe().T

## 8. Distribution & outlier diagnostics

Boxplots per metric (raw units differ, so plotted individually) to flag implausible values before they
reach a risk-indicator calculation.

In [ ]:
numeric_metrics = wide.columns.tolist()
n = len(numeric_metrics)
ncols = 4
nrows = -(-n // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(numeric_metrics):
    sns.boxplot(y=wide[col].dropna(), ax=axes[i], color='#55A868')
    axes[i].set_title(col)
    axes[i].set_ylabel('')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# IQR-based outlier flags per metric
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return s[(s < lower) | (s > upper)]

print("IQR outlier counts per metric:")
for col in numeric_metrics:
    out = iqr_outliers(wide[col].dropna())
    print(f"  {col:22s}: {len(out)} outlier(s)" + (f"  -> values: {sorted(out.tolist())}" if len(out) else ""))

## 9. Plausibility sanity checks

Cross-checking a couple of derivable relationships that a clean dataset should satisfy e.g. BMI should be
internally consistent with height and weight (`bmi = weight / height^2`), and diastolic should not exceed
systolic.

In [ ]:
bmi_check = wide.copy()
bmi_check['bmi_calculated'] = bmi_check['weight'] / (bmi_check['height'] ** 2)
bmi_check['bmi_diff'] = (bmi_check['bmi'] - bmi_check['bmi_calculated']).abs()

print("Rows where recorded bmi differs from calculated bmi by > 0.5:")
mismatch = bmi_check[bmi_check['bmi_diff'] > 0.5][['height', 'weight', 'bmi', 'bmi_calculated', 'bmi_diff']]
print(mismatch if len(mismatch) else "None found — bmi is internally consistent with height/weight.")

print()
bp_violation = wide[wide['diastolic_bp'] >= wide['systolic_bp']]
print("Rows where diastolic_bp >= systolic_bp (physiologically implausible):")
print(bp_violation[['systolic_bp', 'diastolic_bp']] if len(bp_violation) else "None found.")

## 10. Correlation across metrics

Quick look at how the captured metrics relate to one another at the screening grain; useful context before
designing any composite risk indicator.

In [ ]:
corr = wide.corr(numeric_only=True)
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation between metrics (screening-level)')
plt.tight_layout()
plt.show()

## 11. Summary of findings

- **Grain:** table is EAV-style, one row per `(screening_id, metric_code)`; no duplicate `measurement_id`
  or duplicate natural-key pairs found in this extract; pivoting to a wide screening-level table is safe.
- **Completeness:** `numeric_value`, `unit`, `metric_code`, and `captured_at` are fully populated;
  `text_value` is entirely null in this extract (confirm against the contract whether that's expected).
- **Validation status:** check the printed distribution above; filter out anything other than `valid`
  before this feeds a metric calculation.
- **Units:** confirm every metric mapped to exactly one unit (see Section 5); if not, resolve before
  aggregating.
- **Timestamps:** confirm zero unparseable rows and explicit UTC (`Z`) suffixes (Section 6); required for
  safe `rolling()`/`shift()` windowing later.
- **Outliers / plausibility:** review the IQR flags and the BMI/blood-pressure consistency checks above;
  these are the values worth tracing back to source before they inform a risk score.

**Next steps:** compare `metric_code` coverage and validation rules against the Handbook's metric catalogue
and cohort-suppression thresholds before this table is exposed downstream.
